In [ ]:
import os
import os.path as op
import pickle
import sys
import warnings
from collections import defaultdict
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import permutation_test, ttest_rel
from sklearn.exceptions import InconsistentVersionWarning
from sklearn.metrics import balanced_accuracy_score

sys.path.append(op.abspath(op.join(__file__, "../../../..")))
from utils.utils import correlation_score

sns.set_style("ticks")
sns.set_context("talk", font_scale=1.2, rc={"axes.labelpad": 10})
pd.set_option("display.float_format", "{:.3}".format)

warnings.filterwarnings("ignore", category=InconsistentVersionWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:.3}".format)

In [ ]:
ABS_PATH = sys.path[-1]
RESULTS_PATH = op.join(
    ABS_PATH, "validation/3_prediction/3.2_additional_variables/results"
)
FIG_DIR = op.join(ABS_PATH, "validation/3_prediction/3.2_additional_variables/figures")
os.makedirs(FIG_DIR, exist_ok=True)

PALETTE = {
    "Actual": "#283F94",
    "Predicted": "#AE3033",
}

CONTRASTS = (
    "REST",
    "EMOTION FACES-SHAPES",
    "GAMBLING REWARD",
    "LANGUAGE MATH-STORY",
    "RELATIONAL REL",
    "SOCIAL TOM-RANDOM",
    "WM 2BK-0BK",
    "MOTOR AVG",
)

DATASETS = ["ukb_actual", "ukb_predicted"]

SCORE_FUNCS = {
    "fluid": correlation_score,
    "sex": balanced_accuracy_score,
    "strength": correlation_score,
    "overall_health": correlation_score,
}

In [ ]:
## HELPER FUNCTIONS
def statistic(x, y):
    return ttest_rel(x, y).statistic


def perform_ttest(test_scores, permutation_type="samples"):
    """Performs permutated paired t-tests."""
    post_hoc = []
    for pred_cont in np.unique(
        test_scores[test_scores["Actual"] == "Predicted"]["Contrast"]
    ):
        pred_y = test_scores[
            (test_scores["Actual"] == "Predicted")
            & (test_scores["Contrast"] == pred_cont)
        ]["CV Score"].values
        for comp_cont in np.unique(
            test_scores[test_scores["Actual"] == "Actual"]["Contrast"]
        ):
            comp_y = test_scores[
                (test_scores["Actual"] == "Actual")
                & (test_scores["Contrast"] == comp_cont)
            ]["CV Score"].values
            res = permutation_test(
                (pred_y, comp_y),
                statistic,
                vectorized=False,
                permutation_type=permutation_type,
                alternative="two-sided",
                random_state=42,
                n_resamples=1000,
            )
            stat, p_val = res.statistic, res.pvalue
            post_hoc.append(
                {
                    "Predicted Contrast": pred_cont,
                    "Actual Contrast": comp_cont,
                    "t": stat,
                    "p": p_val,
                }
            )

    res = permutation_test(
        (
            test_scores[test_scores["Contrast"] == "CONNECTOME\nD50"][
                "CV Score"
            ].values,
            test_scores[
                (test_scores["Contrast"] == "EMOTION\nFACES-SHAPES")
                & (test_scores["Actual"] == "Actual")
            ]["CV Score"].values,
        ),
        statistic,
        vectorized=False,
        permutation_type=permutation_type,
        alternative="two-sided",
        random_state=42,
        n_resamples=1000,
    )
    stat, p_val = res.statistic, res.pvalue
    post_hoc.append(
        {
            "Predicted Contrast": "CONNECTOME\nD50",
            "Actual Contrast": "EMOTION\nFACES-SHAPES",
            "t": stat,
            "p": p_val,
        }
    )
    return pd.DataFrame(post_hoc)

In [ ]:
# Prepare dataframes for plotting.
def results_files_dict():
    pred_scores = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    dsets = ["ukb_actual", "ukb_pred"]
    for target in SCORE_FUNCS.keys():
        for dset in dsets:
            for cont in CONTRASTS:
                cont = cont.replace(" ", "_").lower()
                try:
                    fname = glob(
                        op.join(
                            RESULTS_PATH,
                            f"{dset}_{target}_{cont}.pkl",
                        )
                    )[0]
                    pred_scores[target] = {dset: {cont: fname}}
                except:
                    pass
    return pred_scores


def compute_scores(pred_scores):
    scores = {}
    for target in pred_scores:
        scores_ = pd.DataFrame()
        for dset in pred_scores[target]:
            for i, task in enumerate(pred_scores[target][dset]):
                pred = pickle.load(open(pred_scores[target][dset][task], "rb"))
                if i == 0:
                    print(
                        f"Dataset: {dset}, Target: {target}, n = {pred[list(pred.keys())[0]][0]["y_true"].shape[0] + pred[list(pred.keys())[0]][0]["y_true_holdout"].shape[0]}"
                    )
                cv_ = [
                    SCORE_FUNCS[target](fold["y_true_holdout"], fold["y_pred_holdout"])
                    for fold in pred[list(pred.keys())[0]]
                ]
                tmp = pd.DataFrame(cv_, columns=["CV Score"]).assign(
                    Contrast=task.replace("_", "\n").upper(),
                    Dataset=dset.replace("_pred", ""),
                )
                tmp["Actual"] = "Actual" if "pred" not in dset else "Predicted"
                scores_ = pd.concat(
                    [
                        scores_,
                        tmp,
                    ],
                    axis=0,
                )
        scores[target] = scores_
    return scores


scores = compute_scores(results_files_dict())

In [ ]:
# Fluid Intelligence
target = "fluid"
plt.figure(figsize=(10, 5), dpi=300)
plot_df = scores[target].dropna()
ax = sns.pointplot(
    data=plot_df,
    x="Contrast",
    y="CV Score",
    hue="Actual",
    errorbar="sd",
    linestyles="",
    palette=PALETTE,
)
ax.set(xticklabels=[], yticklabels=[])
plt.ylim(-0.1, 0.4)  # Set y-axis limits
sns.despine(offset=10, trim=True)
plt.ylabel("Correlation")
plt.xticks(rotation=45)
plt.legend(loc="right", bbox_to_anchor=(1.15, 0.5)).set_visible(False)
plt.setp(ax.lines, alpha=0.75)
plt.savefig(op.join(FIG_DIR, f"{target}.pdf"), bbox_inches="tight")

perform_ttest(plot_df)

In [ ]:
# Sex Classification
target = "sex"
plt.figure(figsize=(10, 5), dpi=300)
plot_df = scores[target].dropna()
ax = sns.pointplot(
    data=plot_df,
    x="Contrast",
    y="CV Score",
    hue="Actual",
    errorbar="sd",
    linestyles="",
    palette=PALETTE,
)
ax.set(xticklabels=[], yticklabels=[])
plt.ylim(0.5, 1)  # Set y-axis limits
sns.despine(offset=10, trim=True)
plt.ylabel("Balanced Accuracy")
plt.xticks(rotation=45)
plt.legend(loc="right", bbox_to_anchor=(1.15, 0.5)).set_visible(False)
plt.setp(ax.lines, alpha=0.75)
plt.savefig(op.join(FIG_DIR, f"{target}.pdf"), bbox_inches="tight")

perform_ttest(plot_df)

In [ ]:
# Grip Strength Regression
target = "strength"
plt.figure(figsize=(10, 5), dpi=300)
plot_df = scores[target].dropna()
ax = sns.pointplot(
    data=plot_df,
    x="Contrast",
    y="CV Score",
    hue="Actual",
    errorbar="sd",
    linestyles="",
    palette=PALETTE,
)
ax.set(xticklabels=[], yticklabels=[])
plt.ylim(0, 0.8)  # Set y-axis limits
sns.despine(offset=10, trim=True)
plt.ylabel("Correlation")
plt.legend(loc="right", bbox_to_anchor=(1.15, 0.5)).set_visible(False)
plt.setp(ax.lines, alpha=0.75)
plt.savefig(op.join(FIG_DIR, f"{target}.pdf"), bbox_inches="tight")

perform_ttest(plot_df)

In [ ]:
# Overall Health Regression
target = "overall_health"
plt.figure(figsize=(10, 5), dpi=300)
plot_df = scores[target].dropna()
ax = sns.pointplot(
    data=plot_df,
    x="Contrast",
    y="CV Score",
    hue="Actual",
    errorbar="sd",
    linestyles="",
    palette=PALETTE,
)
ax.set(xticklabels=[], yticklabels=[])
plt.ylim(-0.2, 0.2)  # Set y-axis limits
plt.yticks(np.arange(-0.2, 0.3, 0.1))  # Set y-axis ticks with increment of 0.1
sns.despine(offset=10, trim=True)
plt.ylabel("Correlation")
plt.xticks(rotation=45)
plt.legend(loc="right", bbox_to_anchor=(1.15, 0.5)).set_visible(False)
plt.setp(ax.lines, alpha=0.75)
plt.savefig(op.join(FIG_DIR, f"{target}.pdf"), bbox_inches="tight")

perform_ttest(plot_df)